In [6]:
! which python && which 

/home/buka2004/PTQ-LLM-MIPT/.venv10/bin/python


# Imports

In [10]:
import torch
import numpy as np
import math
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
import os
from safetensors.torch import load_file

# Add imports for the critical model modification step
from transformers.pytorch_utils import Conv1D
from deepspeed.compression.helper import convert_conv1d_to_linear

import sys
import argparse
import os
from llmcompressor.modifiers.quantization import GPTQModifier
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from llmcompressor import oneshot
from transformers.pytorch_utils import Conv1D
from deepspeed.compression.helper import convert_conv1d_to_linear
from vllm import LLM, SamplingParams
import torch
from torch.utils.data import DataLoader, SequentialSampler
import math
import numpy as np
from transformers import default_data_collator, set_seed
from eval_compressed import evaluate_perplexity


ModuleNotFoundError: No module named 'llmcompressor'

# Apply

In [7]:
def evaluate_perplexity(model, tokenizer, dataset, device='cuda'):
    """
    Calculates perplexity on a given dataset using a sliding window approach.
    """
    print(f'Set model to device: {device}')
    model = model.to(device)

    print("Tokenizing and preparing the dataset...")
    encodings = tokenizer("\n\n".join(dataset["text"]), return_tensors="pt")
    seq_len = encodings.input_ids.size(1)
    
    nlls = [] # Negative Log-Likelihoods
    total_eval_tokens = 0
    
    print(f"Starting perplexity calculation with stride {STRIDE}...")
    prev_end_loc = 0
    
    for begin_loc in tqdm(range(0, seq_len, STRIDE)):
        end_loc = min(begin_loc + MAX_SEQ_LENGTH, seq_len)
        trg_len = end_loc - prev_end_loc  # may be different from stride on last loop
        
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()

        with torch.no_grad():
            outputs = model(input_ids)
            # Shift so that tokens < n predict n
            shift_logits = outputs.logits[..., :-1, :].contiguous()
            shift_labels = target_ids[..., 1:].contiguous()
            
            # Calculate loss only for the new tokens in this window
            loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), 
                           shift_labels.view(-1))
            
            # Reshape loss to match sequence length
            loss = loss.view(shift_labels.size())
            
            # Only take loss for the new tokens (after the overlap)
            if begin_loc > 0:
                # For alldebug  windows except the first, we skip the overlap
                loss = loss[:, (MAX_SEQ_LENGTH - STRIDE):]
            
            # For the last window, we might have fewer tokens
            if loss.size(1) > trg_len - 1:
                loss = loss[:, :trg_len - 1]
            
            neg_log_likelihood = loss.sum()
            num_tokens_in_loss = loss.numel()

        nlls.append(neg_log_likelihood)
        total_eval_tokens += num_tokens_in_loss
        prev_end_loc = end_loc

        if end_loc == seq_len:
            break

    # Calculate perplexity
    total_nll = torch.stack(nlls).sum()
    ppl = torch.exp(total_nll / total_eval_tokens)

    return ppl.item()